# MARKDOWN HÜCRESİ
"""
# Decision Tree (Karar Ağacı) Ödevi
## Dava Sonuçları Tahmin Modeli

Bu çalışmada dava sonuçlarını tahmin etmek için Decision Tree algoritması kullanacağız.

**Görevler:**
1. Veri temizleme (eksik/aykırı değerler)
2. Eğitim-test ayrımı (%80-%20)
3. Model eğitimi
4. Performans metrikleri (Accuracy, Precision, Recall, F1-Score)
5. Model görselleştirme ve yorum
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn kütüphaneleri
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, classification_report, confusion_matrix)
from sklearn.preprocessing import LabelEncoder

# Uyarıları kapat
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
print("✓ Kütüphaneler yüklendi!")

## Adım 1: Veriyi Yükleme ve İlk İnceleme

In [ ]:
# Veri setini yükle
df = pd.read_csv('dava_sonuclari.csv')

print("📊 Veri Seti Boyutu:", df.shape)
print(f"   → {df.shape[0]} satır, {df.shape[1]} sütun\n")

print("="*70)
print("📋 İlk 5 Satır:")
print("="*70)
print(df.head())

print("\n" + "="*70)
print("📝 Veri Tipleri:")
print("="*70)
print(df.dtypes)

print("\n" + "="*70)
print("📈 İstatistiksel Özet:")
print("="*70)
print(df.describe())


## Görev 1: Eksik ve Aykırı Değer Kontrolü

Veri kalitesi model başarısı için kritiktir. Eksik değerleri ve aykırı değerleri kontrol edelim.

In [ ]:
print("🔍 EKSİK DEĞER ANALİZİ")
print("="*70)

# Eksik değer sayıları
eksik_degerler = df.isnull().sum()
print("\n📊 Her Kolondaki Eksik Değer Sayısı:")
print(eksik_degerler)

# Eksik değer yüzdeleri
eksik_yuzde = (df.isnull().sum() / len(df)) * 100
eksik_df = pd.DataFrame({
    'Eksik Sayı': eksik_degerler,
    'Yüzde (%)': eksik_yuzde
})
print("\n📊 Eksik Değer Detayları:")
print(eksik_df[eksik_df['Eksik Sayı'] > 0])

# Eksik değer görselleştirme
if df.isnull().sum().sum() > 0:
    plt.figure(figsize=(12, 5))
    sns.heatmap(df.isnull(), cbar=True, cmap='viridis', yticklabels=False)
    plt.title('Eksik Değer Haritası (Sarı: Eksik, Mor: Dolu)', 
             fontsize=14, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.show()
    
    print("\n⚠️ Eksik değerler tespit edildi!")
    print("   Çözüm: Eksik değerleri içeren satırları çıkaracağız.")
    
    # Eksik değerleri temizle
    df_temiz = df.dropna()
    print(f"\n✓ Temizleme sonrası: {len(df_temiz)} satır kaldı ({len(df) - len(df_temiz)} satır çıkarıldı)")
else:
    df_temiz = df.copy()
    print("\n✓ Eksik değer bulunamadı! Veri seti temiz.")


In [ ]:
print("\n" + "="*70)
print("🔍 AYKIRI DEĞER ANALİZİ")
print("="*70)

# Sayısal kolonları seç
sayisal_kolonlar = df_temiz.select_dtypes(include=[np.number]).columns.tolist()

if len(sayisal_kolonlar) > 0:
    # Her sayısal kolon için boxplot
    fig, axes = plt.subplots(1, len(sayisal_kolonlar), figsize=(5*len(sayisal_kolonlar), 5))
    
    if len(sayisal_kolonlar) == 1:
        axes = [axes]
    
    for i, col in enumerate(sayisal_kolonlar):
        axes[i].boxplot(df_temiz[col].dropna())
        axes[i].set_title(f'{col}\nAykırı Değer Kontrolü', fontweight='bold')
        axes[i].set_ylabel('Değer')
        axes[i].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Aykırı değerler (outliers) boxplot'taki noktalar ile gösterilir.")
    print("   Şimdilik aykırı değerleri koruyacağız, çok uç değerler varsa manuel çıkarılabilir.")
else:
    print("\n✓ Sayısal kolon bulunamadı, aykırı değer analizi yapılamadı.")


## Görev 2: Veriyi Eğitim (%80) ve Test (%20) Olarak Ayırma

Modeli eğitmek için veriyi ikiye ayırmalıyız:
- **Eğitim Seti (%80)**: Model bu veri ile öğrenecek
- **Test Seti (%20)**: Model performansını bu veri ile ölçeceğiz

In [ ]:
print("🔀 VERİ AYIRMA İŞLEMİ")
print("="*70)

# Hedef değişkeni belirle (sonuç/label kolonu)
# NOT: Kendi veri setinizdeki hedef kolon adını yazın
hedef_kolon = 'Sonuc'  # veya 'Karar', 'Durum' vb.

# Hedef kolon var mı kontrol et
if hedef_kolon not in df_temiz.columns:
    print(f"⚠️ UYARI: '{hedef_kolon}' kolonu bulunamadı!")
    print(f"   Mevcut kolonlar: {df_temiz.columns.tolist()}")
    print("\n   Lütfen hedef_kolon değişkenini doğru kolon adı ile değiştirin.")
else:
    # Özellikler (X) ve hedef (y) ayırımı
    X = df_temiz.drop(hedef_kolon, axis=1)
    y = df_temiz[hedef_kolon]
    
    print(f"✓ Hedef değişken: {hedef_kolon}")
    print(f"✓ Özellik sayısı: {X.shape[1]}")
    print(f"✓ Hedef sınıflar: {y.unique()}")
    print(f"✓ Hedef dağılımı:\n{y.value_counts()}")
    
    # Kategorik değişkenleri sayısala çevir
    label_encoders = {}
    for col in X.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le
        print(f"\n✓ '{col}' kategorik kolonu sayısala çevrildi")
    
    # Hedef değişken de kategorikse çevir
    if y.dtype == 'object':
        le_y = LabelEncoder()
        y = le_y.fit_transform(y)
        print(f"\n✓ Hedef değişken sayısala çevrildi: {dict(enumerate(le_y.classes_))}")
    
    # Eğitim-test ayrımı
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, 
        test_size=0.2,      # %20 test
        random_state=42,    # Tekrarlanabilirlik için
        stratify=y          # Sınıf dengesini koru
    )
    
    print("\n" + "="*70)
    print("📊 VERİ AYIRMA SONUÇLARI:")
    print("="*70)
    print(f"✓ Eğitim seti boyutu: {X_train.shape[0]} satır ({X_train.shape[0]/len(df_temiz)*100:.1f}%)")
    print(f"✓ Test seti boyutu  : {X_test.shape[0]} satır ({X_test.shape[0]/len(df_temiz)*100:.1f}%)")
    print(f"✓ Özellik sayısı    : {X_train.shape[1]}")

## Görev 3: Decision Tree Modelini Kurma ve Eğitme

Decision Tree (Karar Ağacı), veriye bakarak karar kuralları oluşturan bir algoritmadır.
Örnek: "Eğer yaş > 30 VE gelir > 50000 ise → Sonuç = Kazanır"

In [ ]:
print("🌳 DECISION TREE MODELİ EĞİTİMİ")
print("="*70)

# Modeli oluştur
model = DecisionTreeClassifier(
    max_depth=5,          # Ağaç derinliği (overfitting'i önler)
    min_samples_split=10, # Dallanma için minimum örnek sayısı
    min_samples_leaf=5,   # Yaprakta minimum örnek sayısı
    random_state=42
)

print("✓ Model parametreleri:")
print(f"   - Maksimum derinlik: {model.max_depth}")
print(f"   - Minimum split: {model.min_samples_split}")
print(f"   - Minimum leaf: {model.min_samples_leaf}")

# Modeli eğit
print("\n⏳ Model eğitiliyor...")
model.fit(X_train, y_train)
print("✓ Model eğitimi tamamlandı!")

# Tahmin yap
print("\n⏳ Tahminler yapılıyor...")
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)
print("✓ Tahminler tamamlandı!")

## Görev 4: Model Performans Metrikleri

Modelin ne kadar başarılı olduğunu ölçmek için çeşitli metrikler kullanacağız:
- **Accuracy**: Doğru tahmin oranı
- **Precision**: Pozitif dediğimizde ne kadar doğru söylüyoruz?
- **Recall**: Gerçek pozitiflerin ne kadarını yakalıyoruz?
- **F1-Score**: Precision ve Recall'ın harmonik ortalaması


In [ ]:
print("📊 MODEL PERFORMANS METRİKLERİ")
print("="*70)

# Eğitim seti performansı
train_accuracy = accuracy_score(y_train, y_pred_train)
print("\n🎯 EĞİTİM SETİ PERFORMANSI:")
print(f"   Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")

# Test seti performansı
test_accuracy = accuracy_score(y_test, y_pred_test)
test_precision = precision_score(y_test, y_pred_test, average='weighted', zero_division=0)
test_recall = recall_score(y_test, y_pred_test, average='weighted', zero_division=0)
test_f1 = f1_score(y_test, y_pred_test, average='weighted', zero_division=0)

print("\n🎯 TEST SETİ PERFORMANSI:")
print(f"   Accuracy : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"   Precision: {test_precision:.4f}")
print(f"   Recall   : {test_recall:.4f}")
print(f"   F1-Score : {test_f1:.4f}")

# Overfitting kontrolü
print("\n🔍 OVERFİTTİNG KONTROLÜ:")
fark = train_accuracy - test_accuracy
if fark > 0.1:
    print(f"   ⚠️ Yüksek fark tespit edildi! ({fark*100:.2f}%)")
    print("   Model eğitim setini ezberliyor olabilir (overfitting).")
else:
    print(f"   ✓ Fark normal seviyede ({fark*100:.2f}%)")

# Detaylı rapor
print("\n" + "="*70)
print("📋 DETAYLI SINIFLANDIRMA RAPORU:")
print("="*70)
print(classification_report(y_test, y_pred_test, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, 
            square=True, linewidths=1, linecolor='black')
plt.title('Confusion Matrix (Karışıklık Matrisi)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Gerçek Değer', fontsize=12)
plt.xlabel('Tahmin Edilen Değer', fontsize=12)
plt.tight_layout()
plt.show()

print("\n💡 Confusion Matrix Açıklaması:")
print("   - Sol üst köşe: Doğru negatif tahminler")
print("   - Sağ alt köşe: Doğru pozitif tahminler")
print("   - Diğer hücreler: Yanlış tahminler")

## Görev 5: Karar Ağacını Görselleştirme ve Yorum

Karar ağacının yapısını görselleştirerek hangi özelliklerin 
daha önemli olduğunu anlayabiliriz.

In [ ]:
print("🌳 KARAR AĞACI GÖRSELLEŞTİRME")
print("="*70)

# Ağaç görselleştirmesi
plt.figure(figsize=(25, 15))
plot_tree(model, 
          feature_names=X.columns,
          class_names=[str(c) for c in model.classes_],
          filled=True,           # Renklendirme
          rounded=True,          # Yuvarlak köşeler
          fontsize=10,
          proportion=True)       # Oran göster
plt.title('